In [58]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split

In [59]:
df = pd.read_csv('/Users/mayurimamdi/Downloads/arXiv_scientific dataset.csv', encoding='latin-1', nrows=100000)

In [60]:
df.head()

,id,title,category,category_code,published_date,updated_date,authors,first_author,summary,summary_word_count
0,cs-9308101v1,Dynamic Backtracking,Artificial Intelligence,cs.AI,8/1/93,8/1/93,['M. L. Ginsberg'],'M. L. Ginsberg',Because of their occasional need to return to ...,79
1,cs-9308102v1,A Market-Oriented Programming Environment and ...,Artificial Intelligence,cs.AI,8/1/93,8/1/93,['M. P. Wellman'],'M. P. Wellman',Market price systems constitute a well-underst...,119
2,cs-9309101v1,An Empirical Analysis of Search in GSAT,Artificial Intelligence,cs.AI,9/1/93,9/1/93,"['I. P. Gent', 'T. Walsh']",'I. P. Gent',We describe an extensive study of search in GS...,167
3,cs-9311101v1,The Difficulties of Learning Logic Programs wi...,Artificial Intelligence,cs.AI,11/1/93,11/1/93,"['F. Bergadano', 'D. Gunetti', 'U. Trinchero']",'F. Bergadano',As real logic programmers normally use cut (!)...,174
4,cs-9311102v1,Software Agents: Completing Patterns and Const...,Artificial Intelligence,cs.AI,11/1/93,11/1/93,"['J. C. Schlimmer', 'L. A. Hermens']",'J. C. Schlimmer',To support the goal of allowing users to recor...,187


In [61]:
df.isnull().sum()

id                    0
title                 0
category              0
category_code         0
published_date        0
updated_date          0
authors               0
first_author          0
summary               0
summary_word_count    0
dtype: int64

In [62]:
df.duplicated().sum()

np.int64(0)

In [63]:
df.shape

(100000, 10)

In [64]:
df['category'].value_counts()

category
Machine Learning                                          36272
Computer Vision and Pattern Recognition                   27945
Artificial Intelligence                                   12497
Machine Learning (Statistics)                              9946
Computation and Language (Natural Language Processing)     3737
                                                          ...  
General Mathematics                                           1
Physics Education                                             1
Cellular Automata and Lattice Gases                           1
Number Theory                                                 1
General Economics                                             1
Name: count, Length: 130, dtype: int64

In [65]:
min_samples = 5000 
df = df[df['category'].map(df['category'].value_counts()) >= min_samples]

In [66]:
df.shape

(86660, 10)

In [69]:
df = df.reset_index(drop=True)
X = df['title'] + ' ' + df['summary']  
y = df['category']

### First lets try with NLP 

In [70]:
corpus = []
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

for i in range(len(X)):
    review = re.sub('[^a-zA-Z]', ' ', X[i])
    review = review.lower().split()
    review = [ps.stem(word) for word in review if word not in stop_words]
    review = ' '.join(review)
    corpus.append(review)

In [71]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(corpus).toarray()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(accuracy_score(y_test, y_pred))


                                         precision    recall  f1-score   support

                Artificial Intelligence       0.72      0.85      0.78      2462
Computer Vision and Pattern Recognition       0.92      0.91      0.91      5590
                       Machine Learning       0.81      0.63      0.71      7257
          Machine Learning (Statistics)       0.43      0.70      0.53      2023

                               accuracy                           0.76     17332
                              macro avg       0.72      0.77      0.73     17332
                           weighted avg       0.79      0.76      0.77     17332

[[2083   56  256   67]
 [ 108 5063  318  101]
 [ 634  306 4596 1721]
 [  65   54  491 1413]]
0.7590006923609508


## Lets try Deep Learning

In [79]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

max_words = 10000
max_len = 150

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(corpus)

X_seq = tokenizer.texts_to_sequences(corpus)
X_pad = pad_sequences(X_seq, maxlen=max_len)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [81]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pad, y, test_size=0.2, random_state=42
)

In [82]:
model = Sequential()

model.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_len))
model.add(LSTM(128))
model.add(Dropout(0.5))

model.add(Dense(64, activation='relu'))
model.add(Dense(4, activation='softmax'))  # 4 classes

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

/opt/homebrew/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [83]:
model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Epoch 1/5
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 115s 104ms/step - accuracy: 0.7457 - loss: 0.6754 - val_accuracy: 0.7779 - val_loss: 0.5735
Epoch 2/5
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 111s 102ms/step - accuracy: 0.7967 - loss: 0.5467 - val_accuracy: 0.7896 - val_loss: 0.5645
Epoch 3/5
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 111s 103ms/step - accuracy: 0.8146 - loss: 0.4976 - val_accuracy: 0.7944 - val_loss: 0.5490
Epoch 4/5
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 113s 105ms/step - accuracy: 0.8349 - loss: 0.4429 - val_accuracy: 0.7918 - val_loss: 0.5748
Epoch 5/5
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 114s 105ms/step - accuracy: 0.8473 - loss: 0.4044 - val_accuracy: 0.7916 - val_loss: 0.5859


In [84]:
model.evaluate(X_test, y_test)

542/542 ━━━━━━━━━━━━━━━━━━━━ 13s 25ms/step - accuracy: 0.7916 - loss: 0.5859


[0.5858770608901978, 0.7915993332862854]